# 🎨 ComfyUI en Google Colab — Drive, modelos e imagen a vídeo

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/I-s-ax/Comfyui_en_colab/blob/main/comfyui_colab_drive.ipynb)

Notebook independiente, preparado para usar ComfyUI desde Android con almacenamiento en Google Drive y formularios de Colab.

**Orden recomendado:** selecciona una GPU en *Entorno de ejecución → Cambiar tipo de entorno*; ejecuta las celdas de arriba hacia abajo; descarga un checkpoint/LoRA y después inicia ComfyUI.

Los campos `#@param` aparecen como formularios. Para generar vídeo también necesitas un modelo de vídeo y un workflow compatible. Este notebook está inspirado en la organización de otros proyectos públicos, pero su código es propio.

## 1. Conectar Google Drive

In [ ]:
#@title 🔌 Conectar Google Drive
from google.colab import drive
from pathlib import Path

MONTAR_DRIVE = True #@param {type:"boolean"}
NOMBRE_CARPETA = "ComfyUI" #@param {type:"string"}

if MONTAR_DRIVE:
    drive.mount("/content/drive")
    WORKSPACE = str(Path("/content/drive/MyDrive") / NOMBRE_CARPETA)
else:
    WORKSPACE = "/content/ComfyUI"

Path(WORKSPACE).mkdir(parents=True, exist_ok=True)
print("Espacio de trabajo:", WORKSPACE)

## 2. Instalar o actualizar ComfyUI

In [ ]:
#@title 🧰 Instalar / actualizar ComfyUI
import subprocess
from pathlib import Path

ACTUALIZAR_COMFYUI = True #@param {type:"boolean"}
repo = Path(WORKSPACE)

if (repo / ".git").exists():
    if ACTUALIZAR_COMFYUI:
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    if any(repo.iterdir()):
        raise RuntimeError(f"{repo} ya existe y no parece una instalación Git. Cambia NOMBRE_CARPETA en la primera celda.")
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", str(repo)], check=True)

subprocess.run(["python", "-m", "pip", "install", "-r", str(repo / "requirements.txt")], check=True)
for name in ["checkpoints", "vae", "loras", "controlnet", "upscale_models", "diffusion_models", "text_encoders", "clip_vision"]:
    (repo / "models" / name).mkdir(parents=True, exist_ok=True)
print("ComfyUI preparado en:", repo)

## 3. Descargar un modelo
Pega un enlace de descarga de Civitai o Hugging Face. Si Civitai requiere token, guárdalo en Colab Secrets con el nombre `CIVITAI_API_TOKEN` o introdúcelo en el campo manual. No compartas el token.

In [ ]:
#@title 📥 Descargar modelo
URL_MODELO = "" #@param {type:"string"}
CARPETA_DESTINO = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "diffusion_models", "text_encoders", "clip_vision"]
NOMBRE_ARCHIVO = "" #@param {type:"string"}
TOKEN_CIVITAI_MANUAL = "" #@param {type:"string"}

import os, re, time, requests
from pathlib import Path
from urllib.parse import urlparse, unquote

if not URL_MODELO.strip():
    raise ValueError("Pega un enlace en URL_MODELO.")

token = TOKEN_CIVITAI_MANUAL.strip()
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("CIVITAI_API_TOKEN") or ""
    except Exception:
        token = ""

original_url = URL_MODELO.strip()
url = original_url
if "civitai.com" in urlparse(url).netloc.lower() and token:
    url += ("&" if "?" in url else "?") + "token=" + token

dest = Path(WORKSPACE) / "models" / CARPETA_DESTINO
dest.mkdir(parents=True, exist_ok=True)
filename = NOMBRE_ARCHIVO.strip() or unquote(Path(urlparse(original_url).path).name)
if not filename or filename.lower() in ("download", "models"):
    filename = "modelo_descargado.safetensors"
target = dest / Path(filename).name

if target.exists() and target.stat().st_size:
    print("Ya existe:", target, f"({target.stat().st_size/1e9:.2f} GB)")
else:
    error = None
    for attempt in range(3):
        temp = target.with_name(target.name + ".part")
        try:
            with requests.get(url, stream=True, allow_redirects=True, timeout=(30, 120),
                              headers={"User-Agent": "ComfyUI-Colab-Downloader/1.0"}) as r:
                if r.status_code != 200:
                    raise RuntimeError(f"HTTP {r.status_code}; revisa enlace, permisos y token.")
                if "text/html" in r.headers.get("Content-Type", "").lower():
                    raise RuntimeError("El servidor devolvió HTML, no el archivo del modelo.")
                cd = r.headers.get("Content-Disposition", "")
                if not NOMBRE_ARCHIVO.strip():
                    m = re.search(r'filename="?([^";]+)', cd, re.I)
                    if m:
                        target = dest / Path(unquote(m.group(1))).name
                        temp = target.with_name(target.name + ".part")
                total = int(r.headers.get("Content-Length", "0") or 0)
                got = 0
                with open(temp, "wb") as f:
                    for chunk in r.iter_content(8 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            got += len(chunk)
                            if total:
                                print(f"Descargado {got/1e9:.2f}/{total/1e9:.2f} GB", end="\r")
                if total and got != total:
                    raise RuntimeError(f"Descarga incompleta ({got}/{total} bytes).")
                with open(temp, "rb") as f:
                    head = f.read(500).lower()
                if b"<html" in head or b"<!doctype" in head:
                    raise RuntimeError("El contenido parece HTML, no un modelo.")
                os.replace(temp, target)
                print(f"\nDescarga completada: {target} ({got/1e9:.2f} GB)")
                error = None
                break
        except Exception as e:
            error = e
            if temp.exists():
                temp.unlink()
            print(f"Intento {attempt+1}/3: {e}")
            if attempt < 2:
                time.sleep(2 * (attempt + 1))
    if error:
        raise RuntimeError(f"No se pudo descargar: {error}") from error
print("Carpeta:", dest)

## 4. Listar modelos

In [ ]:
#@title 🔎 Listar archivos de modelos
MODELOS_A_LISTAR = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "diffusion_models", "text_encoders", "clip_vision"]

from pathlib import Path
folder = Path(WORKSPACE) / "models" / MODELOS_A_LISTAR
folder.mkdir(parents=True, exist_ok=True)
files = sorted(p for p in folder.iterdir() if p.is_file())
print("Carpeta:", folder)
if not files:
    print("No hay archivos en esta carpeta.")
for p in files:
    print(f"{p.name} — {p.stat().st_size/1e9:.2f} GB")

## 5. Iniciar ComfyUI y abrir el enlace

In [ ]:
#@title 🚀 Iniciar ComfyUI y mostrar enlace
import subprocess, time, re, sys, urllib.request
from pathlib import Path

repo = Path(WORKSPACE)
if not (repo / "main.py").exists():
    raise FileNotFoundError("No encuentro main.py. Ejecuta primero la instalación.")

cloudflared = "/usr/local/bin/cloudflared"
if not Path(cloudflared).exists():
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True, check=True)

with open("/content/comfyui.log", "w") as log:
    comfy = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"],
                             cwd=str(repo), stdout=log, stderr=subprocess.STDOUT)

print("Esperando a ComfyUI...")
ready = False
for _ in range(120):
    if comfy.poll() is not None:
        raise RuntimeError("ComfyUI terminó. Revisa /content/comfyui.log")
    try:
        urllib.request.urlopen("http://127.0.0.1:8188", timeout=2)
        ready = True
        break
    except Exception:
        time.sleep(2)
if not ready:
    raise TimeoutError("ComfyUI no respondió en 4 minutos.")

tlog = open("/content/cloudflared.log", "w")
tunnel = subprocess.Popen([cloudflared, "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
                          stdout=tlog, stderr=subprocess.STDOUT)
public_url = None
for _ in range(60):
    time.sleep(2)
    logtext = Path("/content/cloudflared.log").read_text(errors="ignore")
    match = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", logtext)
    if match:
        public_url = match.group(0)
        break
if public_url:
    print("ComfyUI listo:", public_url)
    print("El enlace es temporal y cualquiera que lo tenga podría acceder.")
else:
    print("No se detectó automáticamente el enlace. Registro:")
    print(Path("/content/cloudflared.log").read_text(errors="ignore")[-3000:])

## Solución de problemas
- **403 de Civitai:** verifica el token, los permisos y que el enlace corresponda al archivo correcto.
- **404:** comprueba el ID del modelo y el `fileId`; son identificadores distintos.
- **El modelo no aparece:** confirma la carpeta de destino, que el archivo no termine en `.part`, y refresca o reinicia ComfyUI.
- **T4 sin memoria:** baja la resolución y el batch size; evita cargar varios modelos grandes a la vez.
- **Drive lento:** el almacenamiento temporal de Colab se borra al finalizar la sesión.
- Checkpoints, LoRA, VAE y modelos de vídeo son tipos distintos y necesitan workflows compatibles.

El enlace de Cloudflare es temporal y público. No uses una sesión expuesta para contenido sensible sin añadir protección.